In [ ]:
import io
import torch
import requests
import chess.pgn
from sklearn.model_selection import train_test_split
from transformers import pipeline
from transformers import AutoModelForCausalLM, AutoTokenizer


<h1> Data Fetching</h1>

In [2]:
#Get games from lichess using their rest APIs
url = "https://lichess.org/api/games/user/MashroorNewaz"
params = {"max": 5000}        #Set number of games to retrieve
response = requests.get(url, params=params)      
if response.status_code == 200:
    # Proceed with processing the data
    print("Request successful")
else:
    print(f"Request failed with status code: {response.status_code}")

Request successful


<h1>Parsing & Data Preprocessing</h2>

<h3> Preprocessing Steps </h3>
<div>1. Start & End tags/special tokens</div>
<div>2. tokenization</div>

In [ ]:
#Parsing
games_data = []
games_metadata = []
pgn_data = io.StringIO(response.text)
while True:
    game = chess.pgn.read_game(pgn_data)
    if game is None:
        break
    
    white = game.headers.get("White")
    black = game.headers.get("Black")
    result = game.headers.get("Result")
    print(f"White: {white}, Black: {black}, result = {result}")

    #DATA Preprocessing
    
    # Initialize the board for this game
    board = game.board()
    san_moves = []

    # Iterate through all moves and play them on a board.
    for move in game.mainline_moves():
        #Note: use move.uci() instead of board.san(move) for moves in UCI format
        san_moves.append(board.san(move)) # Convert to SAN string
        board.push(move)                 # Update board state for the next move
    
    # Join with spaces for NLP tokenization
    move_sequence = " ".join(san_moves)
    games_data.append(move_sequence)
    games_metadata.append({
        'white': white,
        'black': black,
        'moves': move_sequence
    })

White: MashroorNewaz, Black: PaulNikol, result = 1-0
White: MashroorNewaz, Black: tigranakert, result = 0-1
White: Marcokings, Black: MashroorNewaz, result = 0-1
White: elshebokshy, Black: MashroorNewaz, result = 0-1
White: MashroorNewaz, Black: abbyjohnsallau, result = 1-0
White: erevan35, Black: MashroorNewaz, result = 1-0
White: u5556581111, Black: MashroorNewaz, result = 1/2-1/2
White: MashroorNewaz, Black: baird17, result = 1-0
White: aharut, Black: MashroorNewaz, result = 1-0
White: mdndra, Black: MashroorNewaz, result = 1-0
White: MashroorNewaz, Black: MeatYou, result = 0-1
White: MashroorNewaz, Black: FatWomanAppreciator, result = 1-0
White: Klaas87, Black: MashroorNewaz, result = 1-0
White: MashroorNewaz, Black: m1stt, result = 1-0
White: akonkel, Black: MashroorNewaz, result = 1-0
White: Serp_76, Black: MashroorNewaz, result = 1/2-1/2
White: charlespalmer, Black: MashroorNewaz, result = 0-1
White: MashroorNewaz, Black: zonderzon, result = 1-0
White: MashroorNewaz, Black: RFRA

<h1>Training</h1>

In [ ]:
from datasets import Dataset
#Split Data into Training + Test
train_data, test_data, train_meta, test_meta = train_test_split(games_data, games_metadata, test_size=0.3, random_state=42)

print(f"Training: {len(train_data)}")
print(f"Testing: {len(test_data)}")

train_dataset = Dataset.from_list([{"text": seq} for seq in train_data])

Training: 956
Testing: 411


In [7]:
device = 0 if torch.cuda.is_available() else -1
model_name = "Waterhorse/chessgpt-base-v1"
#model_name = "derickio/chess-gpt-4.5M"

#Load the model and the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

engine = pipeline(task="text-generation", model=model, tokenizer=tokenizer, device=device)

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

In [ ]:
#Data format change to match 
magnus_formatted_data = [
    f";{game_moves}" 
    for game_moves in train_data
]

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

#Fine Tuning model use LoRA Technique
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query_key_value", "dense"],  # GPT-2 compatible #"query_key_value", "dense"
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

#Applying LoRA to the model
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

training_args = SFTConfig(
    per_device_train_batch_size=4,       
    gradient_accumulation_steps=8,     
    warmup_steps=200,                   
    num_train_epochs=1,             
    learning_rate=2e-4,                   
    fp16=False,  
    bf16=False,
    logging_steps=50,
    output_dir="./magnus-chessGPT",
    save_steps=500,
    dataset_text_field="text",
    packing=True,                         
)

trainer = SFTTrainer(
    model=lora_model,
    train_dataset=train_dataset,
    args=training_args,          # SFTConfig replaces both TrainingArguments and SFTTrainer params
)

trainer.train()

trainable params: 7,864,320 || all params: 2,783,728,640 || trainable%: 0.2825


Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-f

Adding EOS to train dataset:   0%|          | 0/956 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/956 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/956 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.
c:\Users\mashr\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


<h1>Model Evaluation</h1>

In [ ]:
def extract_player_moves (game_meta, player_name="MashroorNewaz"):
    white = game_meta['white']
    black = game_meta['black']
    moves = game_meta['moves'].split()

    # Determine which color the player was
    if white == player_name:
        color = "white"
        start_idx = 0 
    elif black == player_name:
        color = "black"
        start_idx = 1
    else:
        return []
    
    player_positions = []

    for i in range(start_idx, len(moves), 2):
        context = " ".join(moves[:i])       #position
        target_move = moves[i]              #move made by player

        player_positions.append((context, target_move))

    return player_positions

def prepare_evaluation_data(test_metadata, player_name="MashroorNewaz"):
    """Prepare all test positions for evaluation."""
    all_positions = []

    for game_meta in test_metadata:
        positions = extract_player_moves(game_meta, player_name)
        all_positions.append(positions)

    return all_positions

def calculate_move_accuracy (model, tokenizer, positions, k=1):
    correct = 0
    total = len(positions)

    device = next(model.parameters()).device    #
    model.eval()            #Set model in evaluation mode

    for context, actual_move in tqdm(positions, desc=f"Top-{k} Accuracy"):
        inputs = tokenizer(context, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        # Generate top-k predictions
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                num_beams=k,
                num_return_sequences=k,
                early_stopping=True,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False  
            )
        
        # Decode predictions
        predictions = []
        for output in outputs:
            new_tokens = output[inputs['input_ids'].shape[1]:]
            pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            first_move = pred_text.split()[0] if pred_text else ""
            predictions.append(first_move)
        
        # Check if actual move is in top-k predictions
        if actual_move in predictions:
            correct += 1
    
    accuracy = correct / total if total > 0 else 0
    return accuracy, total

In [ ]:
print("EVALUATION STARTED")

player_name = "MashroorNewaz"
positions = prepare_evaluation_data(test_meta, player_name)
print(f"Total positions to evaluate: {len(positions)}")

# 1. Move Accuracy
print("Calculating Top-1 Move Accuracy...")
top1_acc, n_pos = calculate_move_accuracy(lora_model, tokenizer, positions, k=1)
    
print("Calculating Top-3 Move Accuracy...")
top3_acc, _ = calculate_move_accuracy(lora_model, tokenizer, positions, k=3)

print("EVALUATION RESULTS")
print(f"Positions Evaluated:  {n_pos}")
print(f"Top-1 Move Accuracy:  {top1_acc:.2%} ({int(top1_acc * n_pos)}/{n_pos} correct)")
print(f"Top-3 Move Accuracy:  {top3_acc:.2%}")